# Module 08, Unit 01 — Problem Setup & Probabilistic Model

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 08 | Unit 01

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical threads** | `[GLM]` · `[BAY]` · `[PD]` · `[MLE]` |
| **Prerequisites** | All of Modules 00–07 |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] State the Bayesian logistic regression model as a prior, likelihood, and posterior
- [ ] Write the ELBO as the sum of an expected log-likelihood and a negative KL divergence
- [ ] Derive the KL term in closed form using the Gaussian KL formula from Module 06
- [ ] Explain why the expected log-likelihood requires Monte Carlo estimation
- [ ] Identify which tool from each prior module appears in the model

---

## How to Use This Unit

This unit contains **no code**. Every expression should be derived by hand before reading further. If you can write the ELBO from memory after working through this unit, you are ready for Unit 02.

The Iris dataset is used throughout — not because it is challenging, but because it is familiar. You will recognize the class structure and can verify results visually. The mathematics is the same for any binary classification problem.

---

## 1. The Problem

We observe $n$ labeled examples $(\boldsymbol{x}_1, y_1), \ldots, (\boldsymbol{x}_n, y_n)$ where $\boldsymbol{x}_i \in \mathbb{R}^p$ is a feature vector and $y_i \in \{0, 1\}$ is a binary label. We want to:

1. Learn a classifier that predicts $P(y = 1 \mid \boldsymbol{x})$
2. **Quantify uncertainty** in our predictions — not just give a point estimate, but a distribution over possible classifiers

**The Iris problem.** We use two classes: Setosa ($y=0$) and Versicolor ($y=1$). Features: sepal length ($x_1$) and sepal width ($x_2$), plus a bias term, giving $p = 3$.

The design matrix is:

$$
\boldsymbol{X} = \begin{pmatrix} 1 & x_{1,1} & x_{1,2} \\ 1 & x_{2,1} & x_{2,2} \\ \vdots & \vdots & \vdots \\ 1 & x_{n,1} & x_{n,2} \end{pmatrix} \in \mathbb{R}^{n \times 3}, \qquad \boldsymbol{y} = \begin{pmatrix} y_1 \\ \vdots \\ y_n \end{pmatrix} \in \{0,1\}^n
$$

The classifier weights are $\boldsymbol{w} = (w_0, w_1, w_2)^{\top} \in \mathbb{R}^3$, where $w_0$ is the bias, $w_1$ is the sepal length coefficient, and $w_2$ is the sepal width coefficient.

---

## 2. The Likelihood

Logistic regression models the class probability using the sigmoid function (Module 00 Unit 03, Module 03 Unit 02):

$$
P(y_i = 1 \mid \boldsymbol{x}_i, \boldsymbol{w}) = \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i) = \frac{1}{1 + e^{-\boldsymbol{w}^{\top}\boldsymbol{x}_i}}
$$

Assuming the $n$ observations are conditionally independent given $\boldsymbol{w}$, the likelihood is:

$$
p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w}) = \prod_{i=1}^n \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i)^{y_i}\left(1 - \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i)\right)^{1-y_i}
$$

The **log-likelihood** is the binary cross-entropy (Module 03 Unit 02, Section 4.2):

$$
\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w}) = \sum_{i=1}^n \left[y_i \log\sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i) + (1-y_i)\log(1 - \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i))\right]
$$

> **Recall from Module 03 Unit 02.** The gradient of the log-likelihood with respect to $\boldsymbol{w}$ is:
> $$\nabla_{\boldsymbol{w}}\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w}) = \boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}})$$
> where $\hat{\boldsymbol{p}} = \sigma(\boldsymbol{X}\boldsymbol{w}) \in \mathbb{R}^n$ is the vector of predicted probabilities. This residual structure — predicted minus observed — appears again in the ELBO gradient.

---

## 3. The Prior

The Bayesian approach places a prior distribution over the weights $\boldsymbol{w}$ before seeing any data. We choose an isotropic Gaussian prior:

$$
p(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{0},\, \sigma_0^2 \boldsymbol{I})
$$

where $\sigma_0^2 > 0$ is the prior variance (a hyperparameter). Writing out the density:

$$
\log p(\boldsymbol{w}) = -\frac{p}{2}\log(2\pi\sigma_0^2) - \frac{\|\boldsymbol{w}\|^2}{2\sigma_0^2}
$$

**Why Gaussian?** The Gaussian prior is:
- The maximum entropy distribution for a given mean and variance (Module 04 Unit 03)
- Conjugate to itself under Gaussian likelihoods (Module 06 Unit 02)
- Equivalent to Ridge regularization with $\lambda = 1/\sigma_0^2$ (Module 04 Unit 02, Module 06 Unit 02)

When $\sigma_0^2 \to \infty$ (flat prior), the Bayesian solution converges to the MLE — the MAP estimate $\hat{\boldsymbol{w}}_{\text{MAP}} = \arg\max_{\boldsymbol{w}}[\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w}) + \log p(\boldsymbol{w})]$ with a Gaussian prior gives exactly the Ridge-penalized log-likelihood.

---

## 4. The Posterior and the Inference Problem

By Bayes' theorem, the posterior over weights given data is:

$$
p(\boldsymbol{w} \mid \boldsymbol{X}, \boldsymbol{y}) = \frac{p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\,p(\boldsymbol{w})}{p(\boldsymbol{y} \mid \boldsymbol{X})}
$$

where the **evidence** (also called the marginal likelihood) is:

$$
p(\boldsymbol{y} \mid \boldsymbol{X}) = \int_{\mathbb{R}^p} p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\,p(\boldsymbol{w})\,d\boldsymbol{w}
$$

**The problem.** Because the likelihood involves the sigmoid function — a nonlinear, non-Gaussian function of $\boldsymbol{w}$ — the evidence integral has no closed form. The posterior $p(\boldsymbol{w} \mid \boldsymbol{X}, \boldsymbol{y})$ is not Gaussian and cannot be computed exactly.

**The solution: variational inference.** We approximate the true (intractable) posterior with a tractable distribution $q(\boldsymbol{w})$ from a parametric family, and choose $q$ to be as close as possible to the true posterior in KL divergence.

---

## 5. The Variational Family

We choose a **mean-field Gaussian** as the variational family:

$$
q(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m},\, \text{diag}(\boldsymbol{s}^2))
$$

where:
- $\boldsymbol{m} = (m_0, m_1, m_2)^{\top} \in \mathbb{R}^3$ is the **variational mean** — a free parameter
- $\boldsymbol{s} = (s_0, s_1, s_2)^{\top} \in \mathbb{R}^3_{>0}$ is the **variational standard deviation** — a free parameter
- $\text{diag}(\boldsymbol{s}^2) = \text{diag}(s_0^2, s_1^2, s_2^2)$ is a diagonal covariance matrix

The diagonal structure (mean-field) assumes the posterior weights are approximately independent. This is a simplification — the true posterior has correlations between $w_0$, $w_1$, and $w_2$ — but it makes the optimization tractable.

The log density of $q$ is:

$$
\log q(\boldsymbol{w}) = -\frac{p}{2}\log(2\pi) - \sum_{j=0}^{p-1}\log s_j - \frac{1}{2}\sum_{j=0}^{p-1}\frac{(w_j - m_j)^2}{s_j^2}
$$

**Parameterization.** To avoid the constraint $s_j > 0$ during optimization, we parameterize with $\rho_j = \log s_j \in \mathbb{R}$, so $s_j = e^{\rho_j}$. We optimize over $(\boldsymbol{m}, \boldsymbol{\rho})$ without any positivity constraints.

---

## 6. The Evidence Lower Bound

From Module 07 Unit 03 (Section 6.2), the log evidence decomposes exactly as:

$$
\log p(\boldsymbol{y} \mid \boldsymbol{X}) = \underbrace{\mathbb{E}_{q}\!\left[\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\right] - D_{\text{KL}}(q(\boldsymbol{w}) \| p(\boldsymbol{w}))}_{\text{ELBO}(\boldsymbol{m}, \boldsymbol{\rho})} + \underbrace{D_{\text{KL}}(q(\boldsymbol{w}) \| p(\boldsymbol{w} \mid \boldsymbol{X}, \boldsymbol{y}))}_{\geq\, 0}
$$

Since the log evidence is fixed (it does not depend on $q$), **maximizing the ELBO** is equivalent to **minimizing $D_{\text{KL}}(q \| p_{\text{post}})$** — driving $q$ toward the true posterior.

### 6.1 The ELBO Written Out

$$
\boxed{\text{ELBO}(\boldsymbol{m}, \boldsymbol{\rho}) = \underbrace{\mathbb{E}_{q}\!\left[\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\right]}_{\text{expected log-likelihood}} - \underbrace{D_{\text{KL}}(q(\boldsymbol{w}) \| p(\boldsymbol{w}))}_{\text{KL from prior}}}
$$

Each term plays a distinct role:
- **Expected log-likelihood**: rewards $q$ for placing mass on weights that explain the data well — a measure of fit
- **KL from prior**: penalizes $q$ for deviating from the prior — a regularizer

This is the same bias-variance structure as Ridge regression: the Ridge penalty $\lambda\|\boldsymbol{\beta}\|^2$ is the log-prior under $\boldsymbol{w} \sim \mathcal{N}(\boldsymbol{0}, \sigma_0^2\boldsymbol{I})$ with $\lambda = 1/(2\sigma_0^2)$.

---

## 7. The KL Term — Closed Form

The KL divergence between two Gaussians has a closed form (Module 07 Unit 03, Section 6.4; Module 06 Unit 03 extended to this setting).

For $q(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m}, \text{diag}(\boldsymbol{s}^2))$ and $p(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{0}, \sigma_0^2\boldsymbol{I})$:

$$
D_{\text{KL}}(q \| p) = \frac{1}{2}\sum_{j=0}^{p-1}\left[\frac{s_j^2}{\sigma_0^2} + \frac{m_j^2}{\sigma_0^2} - 1 - \log\frac{s_j^2}{\sigma_0^2}\right]
$$

**Derivation.** Using the general formula $D_{\text{KL}}(\mathcal{N}(\boldsymbol{\mu}_1, \boldsymbol{\Sigma}_1) \| \mathcal{N}(\boldsymbol{\mu}_2, \boldsymbol{\Sigma}_2))$ with $\boldsymbol{\Sigma}_1 = \text{diag}(\boldsymbol{s}^2)$, $\boldsymbol{\Sigma}_2 = \sigma_0^2\boldsymbol{I}$, $\boldsymbol{\mu}_1 = \boldsymbol{m}$, $\boldsymbol{\mu}_2 = \boldsymbol{0}$:

$$
D_{\text{KL}}(q \| p) = \frac{1}{2}\left[\text{tr}(\sigma_0^{-2}\text{diag}(\boldsymbol{s}^2)) + \sigma_0^{-2}\boldsymbol{m}^{\top}\boldsymbol{m} - p + \log\frac{\sigma_0^{2p}}{\prod_j s_j^2}\right]
$$

Since $\text{tr}(\sigma_0^{-2}\text{diag}(\boldsymbol{s}^2)) = \sigma_0^{-2}\sum_j s_j^2$ and $\log(\sigma_0^{2p}/\prod_j s_j^2) = \sum_j \log(\sigma_0^2/s_j^2)$, this simplifies to the per-component sum above.

**In terms of $\rho_j = \log s_j$:**

$$
D_{\text{KL}}(q \| p) = \frac{1}{2}\sum_{j=0}^{p-1}\left[\frac{e^{2\rho_j}}{\sigma_0^2} + \frac{m_j^2}{\sigma_0^2} - 1 - 2\rho_j + \log\sigma_0^2\right]
$$

The $\log\sigma_0^2$ terms are constants (they do not affect the optimization over $\boldsymbol{m}$ and $\boldsymbol{\rho}$) and can be absorbed into a constant offset.

---

## 8. The Expected Log-Likelihood — Why Monte Carlo is Needed

The expected log-likelihood is:

$$
\mathbb{E}_{q}\!\left[\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\right] = \int_{\mathbb{R}^p} \log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\,q(\boldsymbol{w})\,d\boldsymbol{w}
$$

For logistic regression:

$$
= \int_{\mathbb{R}^p} \sum_{i=1}^n \left[y_i \log\sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i) + (1-y_i)\log(1 - \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i))\right] \mathcal{N}(\boldsymbol{w}; \boldsymbol{m}, \text{diag}(\boldsymbol{s}^2))\,d\boldsymbol{w}
$$

**This integral has no closed form.** The sigmoid $\sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i) = 1/(1+e^{-\boldsymbol{w}^{\top}\boldsymbol{x}_i})$ is nonlinear in $\boldsymbol{w}$, so it does not combine with the Gaussian $q(\boldsymbol{w})$ into anything analytically tractable.

**The solution: Monte Carlo estimation.** Using $K$ samples $\boldsymbol{w}^{(k)} \sim q$:

$$
\mathbb{E}_{q}\!\left[\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\right] \approx \frac{1}{K}\sum_{k=1}^K \log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w}^{(k)})
$$

We can evaluate the log-likelihood exactly at any sample $\boldsymbol{w}^{(k)}$ — the only question is how to differentiate through the sampling step to get gradients for optimization. The **reparameterization trick** (Unit 02) solves this.

---

## 9. The Complete Model — Summary

Writing everything together:

$$
\boxed{
\begin{aligned}
&\textbf{Prior:} && \boldsymbol{w} \sim \mathcal{N}(\boldsymbol{0}, \sigma_0^2\boldsymbol{I}) \\
&\textbf{Likelihood:} && y_i \mid \boldsymbol{x}_i, \boldsymbol{w} \sim \text{Bernoulli}(\sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i)) \\
&\textbf{Variational posterior:} && q(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m}, \text{diag}(e^{2\boldsymbol{\rho}})) \\
&\textbf{ELBO:} && \text{ELBO} = \mathbb{E}_q[\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w})] - D_{\text{KL}}(q \| p) \\
&\textbf{Objective:} && \max_{\boldsymbol{m},\, \boldsymbol{\rho}}\;\text{ELBO}(\boldsymbol{m}, \boldsymbol{\rho})
\end{aligned}
}
$$

**Thread connections:**

| ELBO component | Statistical thread | Module origin |
|---|---|---|
| $\log p(\boldsymbol{y}\|\boldsymbol{X},\boldsymbol{w})$ | `[GLM]` — logistic regression | Module 03 Unit 02 |
| $p(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{0}, \sigma_0^2\boldsymbol{I})$ | `[BAY]` — Gaussian prior | Module 06 Unit 01 |
| $D_{\text{KL}}(q \| p)$ | `[BAY]` · `[PD]` — KL closed form | Module 07 Unit 03 |
| $\text{ELBO} \leq \log p(\boldsymbol{y}\|\boldsymbol{X})$ | `[MLE]` — bound on evidence | Module 07 Unit 03 |
| $\max \text{ELBO}$ | `[GLM]` · `[BAY]` — gradient ascent | Module 04 Unit 02 |

---

## Python: Data Exploration and Model Verification

Unit 01 is primarily mathematical. The Python section here loads the Iris dataset, visualizes the classification problem, and verifies the KL formula from Section 7 numerically — confirming the closed-form expression before using it in Unit 02.

In [ ]:
import subprocess, sys
for pkg in ["numpy", "matplotlib", "scipy", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (9,6), 'font.size': 11,
                     'axes.titlesize': 13, 'lines.linewidth': 2, 'figure.dpi': 120})
TS_BLUE='#1e64b4'; TS_AMBER='#c87814'; TS_GREEN='#1e8c50'
TS_GRAY='#64646e'; TS_RED='#b41e1e'; TS_LIGHT='#f5f7fa'
print('Setup complete.')

### Section 1 — Data Setup and Visualization

In [ ]:
# ============================================================
# Load and prepare Iris — Setosa (0) vs Versicolor (1)
# Features: sepal length (col 0) and sepal width (col 1)
# ============================================================
iris = load_iris()
# Select first two classes (Setosa=0, Versicolor=1)
mask = iris.target < 2
X_raw = iris.data[mask, :2]   # sepal length, sepal width
y = iris.target[mask].astype(float)

# Standardize features (zero mean, unit variance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Add bias column
n = len(y)
X = np.column_stack([np.ones(n), X_scaled])   # (100, 3)
p_dim = X.shape[1]   # = 3

print(f'Dataset: {n} observations, {p_dim} features (including bias)')
print(f'Class balance: {int(y.sum())} Versicolor, {int(n-y.sum())} Setosa')
print(f'X shape: {X.shape},  y shape: {y.shape}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for c, col, lab in [(0, TS_BLUE, 'Setosa (y=0)'), (1, TS_AMBER, 'Versicolor (y=1)')]:
    idx = y == c
    ax.scatter(X_raw[idx, 0], X_raw[idx, 1], color=col, alpha=0.7,
               s=55, label=lab, edgecolors='white', linewidths=0.5)
ax.set_xlabel('Sepal length (cm)'); ax.set_ylabel('Sepal width (cm)')
ax.set_title('Iris: Setosa vs Versicolor (raw features)')
ax.legend(fontsize=10)

ax2 = axes[1]
for c, col, lab in [(0, TS_BLUE, 'Setosa (y=0)'), (1, TS_AMBER, 'Versicolor (y=1)')]:
    idx = y == c
    ax2.scatter(X_scaled[idx, 0], X_scaled[idx, 1], color=col, alpha=0.7,
                s=55, label=lab, edgecolors='white', linewidths=0.5)
ax2.set_xlabel('Sepal length (standardized)')
ax2.set_ylabel('Sepal width (standardized)')
ax2.set_title('Standardized features (used in model)')
ax2.legend(fontsize=10)
ax2.axhline(0, color=TS_GRAY, lw=0.8, alpha=0.5)
ax2.axvline(0, color=TS_GRAY, lw=0.8, alpha=0.5)

plt.suptitle('Iris Dataset — Binary Classification Problem', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Section 2 — KL Formula Verification

Verify the closed-form KL formula from Section 7 by comparing it to a Monte Carlo estimate. This confirms the formula before using it in the ELBO optimization.

In [ ]:
# ============================================================
# KL formula verification
# KL(N(m, diag(s²)) || N(0, σ₀²I))
# = (1/2) Σⱼ [sⱼ²/σ₀² + mⱼ²/σ₀² - 1 - log(sⱼ²/σ₀²)]
# ============================================================

def kl_closed_form(m, log_s, sigma0_sq):
    """Closed-form KL(q||p) where q=N(m, diag(exp(2*log_s)))
       and p=N(0, sigma0_sq * I)."""
    s_sq = np.exp(2 * log_s)
    return 0.5 * np.sum(s_sq/sigma0_sq + m**2/sigma0_sq - 1 - np.log(s_sq/sigma0_sq))

def kl_monte_carlo(m, log_s, sigma0_sq, N=200_000, rng=None):
    """Monte Carlo estimate of KL(q||p) = E_q[log q - log p]."""
    if rng is None:
        rng = np.random.default_rng(0)
    s = np.exp(log_s)
    # Draw w ~ q = N(m, diag(s²))
    eps = rng.standard_normal((N, len(m)))
    w_samples = m + s * eps
    # log q(w)
    log_q = stats.multivariate_normal(
        mean=m, cov=np.diag(s**2)).logpdf(w_samples)
    # log p(w)
    log_p = stats.multivariate_normal(
        mean=np.zeros(len(m)), cov=sigma0_sq*np.eye(len(m))).logpdf(w_samples)
    return np.mean(log_q - log_p)

rng = np.random.default_rng(42)
sigma0_sq = 1.0

# Test several (m, log_s) configurations
configs = [
    (np.array([0.5, -0.3, 1.0]),  np.array([-0.5, 0.2, -0.8])),
    (np.array([0.0,  0.0, 0.0]),  np.array([0.0,  0.0,  0.0])),   # q = p: KL should = 0
    (np.array([2.0, -1.0, 0.5]),  np.array([0.5,  0.3, -0.2])),
]

print(f'{"Config":<35} {"Closed form":>14} {"Monte Carlo":>13} {"Error":>10}')
print('-'*75)
for m_test, log_s_test in configs:
    kl_cf = kl_closed_form(m_test, log_s_test, sigma0_sq)
    kl_mc = kl_monte_carlo(m_test, log_s_test, sigma0_sq, rng=rng)
    label = f'm={np.round(m_test,1)}, ρ={np.round(log_s_test,1)}'
    print(f'{label:<35} {kl_cf:>14.6f} {kl_mc:>13.6f} {abs(kl_cf-kl_mc):>10.2e}')

print('\nAll KL values ≥ 0:', all(
    kl_closed_form(m,ls,sigma0_sq) >= -1e-12 for m,ls in configs))

**What do you see?** The closed-form KL and the Monte Carlo estimate agree to within $10^{-3}$ for all configurations. When $q = p$ (zero mean, unit standard deviation matching the prior with $\sigma_0^2 = 1$), the KL is zero — $q$ and $p$ are identical, so the divergence is zero. All KL values are non-negative, consistent with Gibbs' inequality.

**Up next:** Unit 02 derives the gradients of the ELBO with respect to $\boldsymbol{m}$ and $\boldsymbol{\rho}$ using the reparameterization trick, and implements gradient ascent from scratch.

In [ ]:
# Extension workspace
# 1. Verify the KL formula algebraically by expanding
#    E_q[log q(w)] and E_q[log p(w)] separately using the
#    entropy formula for a Gaussian: H(N(m,S)) = (1/2)log|2πeS|
#    Then confirm KL = E_q[log q] - E_q[log p].
#
# 2. Plot the KL as a function of σ₀² (prior variance) for fixed
#    q parameters. What happens as σ₀² → ∞ (flat prior)?
#    What happens as σ₀² → 0 (very tight prior)?
#
# 3. MAP estimate: the MAP solution sets σ₀² to a fixed value and
#    maximizes log p(y|X,w) + log p(w) directly. Show this is
#    equivalent to Ridge logistic regression with λ = 1/σ₀².
#    Implement gradient ascent for MAP and compare to Unit 02's
#    variational solution.


---

## Summary

| Component | Expression | Where derived |
|---|---|---|
| Prior | $p(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{0}, \sigma_0^2\boldsymbol{I})$ | Module 06 Unit 01 |
| Likelihood | $p(\boldsymbol{y}\|\boldsymbol{X},\boldsymbol{w}) = \prod_i \sigma(\boldsymbol{w}^{\top}\boldsymbol{x}_i)^{y_i}(1-\sigma)^{1-y_i}$ | Module 03 Unit 02 |
| Variational $q$ | $q(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m}, \text{diag}(e^{2\boldsymbol{\rho}}))$ | This unit |
| KL closed form | $\frac{1}{2}\sum_j[e^{2\rho_j}/\sigma_0^2 + m_j^2/\sigma_0^2 - 1 - 2\rho_j + \log\sigma_0^2]$ | Module 07 Unit 03 |
| Expected LL | Requires Monte Carlo — no closed form | This unit |
| ELBO | $\mathbb{E}_q[\log p(\boldsymbol{y}\|\boldsymbol{X},\boldsymbol{w})] - D_{\text{KL}}(q\|p)$ | Module 07 Unit 03 |

**Up next:** Unit 02 — Variational Inference: Derivation & Implementation

---
*Module 08, Unit 01 — Threat Surfaces | fischer³ Education*